-  Incremental Processing for the three simpler : external sources - POSITION, CASH, REFERENCE. 
- This notebook reads -> each source's already-cleaned Silver table (built by 07/08/09) and
 MERGEs it into a "_current" table using the shared merge_into_silver() helper
- So that  running the same data twice does notduplicate anything.

In [0]:
%run ./silver_common

In [0]:
from pyspark.sql import functions as F


## A. POSITION

In [0]:
position_df = spark.table(silver_table("position"))
print(f"silver.position row count: {position_df.count()}")

POSITION_MERGE_KEYS = ["internal_fund_id", "internal_asset_id", "business_date"]

break_keys_df = (
    position_df
    .withColumn(
        "_pos_hash",
        F.sha2(F.concat_ws("||", F.col("quantity").cast("string"), F.col("price").cast("string")), 256)
    )
    .groupBy(*POSITION_MERGE_KEYS)
    .agg(F.countDistinct("_pos_hash").alias("_distinct_val_count"))
    .filter(F.col("_distinct_val_count") > 1)
    .select(*POSITION_MERGE_KEYS)
)

break_key_count = break_keys_df.count()
if break_key_count > 0:
    print(
        f"WARNING: {break_key_count} fund/asset/day key(s) have an unresolved "
        f"POSITION_BREAK and are EXCLUDED from position_current until resolved "
        f"(see assumption note above)."
    )

position_clean_df = position_df.join(break_keys_df, on=POSITION_MERGE_KEYS, how="left_anti")

merge_into_silver(position_clean_df, "position_current", merge_key_cols=POSITION_MERGE_KEYS)

position_current_df = spark.table(silver_table("position_current"))
print(f"silver.position_current row count after merge: {position_current_df.count()}")

silver.position row count: 20
silver.position_current row count after merge: 20


**Idempotency check 


## B. CASH

In [0]:
cash_df = spark.table(silver_table("cash"))
print(f"silver.cash row count: {cash_df.count()}")

CASH_MERGE_KEYS = ["cash_id"]
merge_into_silver(cash_df, "cash_current", merge_key_cols=CASH_MERGE_KEYS)

cash_current_df = spark.table(silver_table("cash_current"))
print(f"silver.cash_current row count after merge: {cash_current_df.count()}")

silver.cash row count: 10
silver.cash_current row count after merge: 10


**Idempotency check B:** re-run the two cells above a second time.
Row count must stay identical.

# SECTION C - REFERENCE
- Merge key: internal_asset_id + business_date (matches 09's dedup key).
- Unlike POSITION/CASH, REFERENCE genuinely has multiple versions per asset
- over time (the AST005 currency-flip case) - the MERGE here still upserts
- per (asset, date), so each day's value is its own row, exactly matching
- what 09_silver_external_reference already produces. The "changed" flag
- from 09 stays intact through the merge since it's just another column.


## C. REFERENCE

In [0]:
reference_df = spark.table(silver_table("reference"))
print(f"silver.reference row count: {reference_df.count()}")

REFERENCE_MERGE_KEYS = ["internal_asset_id", "business_date"]
merge_into_silver(reference_df, "reference_current", merge_key_cols=REFERENCE_MERGE_KEYS)

reference_current_df = spark.table(silver_table("reference_current"))
print(f"silver.reference_current row count after merge: {reference_current_df.count()}")

silver.reference row count: 19
silver.reference_current row count after merge: 19


**Idempotency check C:** re-run the two cells above a second time.
Row count must stay identical.

### Summary
All three `_current` tables now exist and are safe to re-run against
without duplicating.

In [0]:
print("Day-7 Incremental Processing summary:")
print(f"  position_current  : {position_current_df.count()} rows")
print(f"  cash_current      : {cash_current_df.count()} rows")
print(f"  reference_current : {reference_current_df.count()} rows")

Day-7 Incremental Processing summary:
  position_current  : 20 rows
  cash_current      : 10 rows
  reference_current : 19 rows
